# Generative Models: From Diffusion to Flow Matching
### A Hands-On Workshop on Modern Generative AI

---

**Duration:** ~3 hours | **Level:** Intermediate

In this workshop, we will build generative models from scratch, starting with simple 2D examples and scaling up to text-conditioned image generation.

**What you will learn:**
1. **DDPM** (Denoising Diffusion Probabilistic Models) -- the foundation of modern image generation
2. **Flow Matching** -- a simpler, faster alternative to diffusion
3. **Conditional Generation** -- generating images from text descriptions using Pokemon data

**Prerequisites:** Basic PyTorch, familiarity with neural networks

## Table of Contents

| Part | Topic | Key Concepts |
|------|-------|-------------|
| **1** | [The Big Picture](#part1) | What generative models do, noise-to-data paradigm |
| **2** | [DDPM: Denoising Diffusion](#part2) | Forward process, reverse process, noise prediction |
| **3** | [Flow Matching](#part3) | Velocity fields, ODE integration, straight paths |
| **4** | [Conditional Image Generation](#part4) | Text conditioning, cross-attention, classifier-free guidance |

---

In [ ]:
# ============================================================
# Setup: Install and import everything we need
# ============================================================
# Uncomment the line below if running on Google Colab
# !pip install -q diffusers transformers datasets accelerate peft matplotlib ipywidgets

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.animation import FuncAnimation
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import Image, HTML, display
from sklearn.datasets import make_moons
import warnings
warnings.filterwarnings('ignore')

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

<a id="part1"></a>
# Part 1: The Big Picture -- What Are Generative Models?

## The Core Idea

**Generative models learn to transform simple noise into complex data.**

Think of it this way:
- We know how to sample from a **simple distribution** (e.g., Gaussian noise -- just call `torch.randn()`)
- We have **examples** of a complex distribution (e.g., images of faces, molecules, music)
- We want to learn a **transformation** that maps noise $\to$ data

```
 Noise (easy to sample)          Data (hard to sample)
 ┌─────────────────┐             ┌─────────────────┐
 │  ·  · · ·  ·    │             │    🌙🌙          │
 │ ·  ·  · ·  · ·  │  ──────►   │  🌙    🌙        │
 │  · · ·  · ·  ·  │  learned   │        🌙🌙      │
 │ · ·  · · · ·  · │  transform │  🌙🌙    🌙      │
 │  ·  · ·  · · ·  │             │    🌙🌙🌙        │
 └─────────────────┘             └─────────────────┘
    x ~ N(0, I)                    x ~ p_data
```

The two main approaches we'll study differ in **how they define this transformation**:

| Approach | Metaphor | How it works |
|----------|----------|-------------|
| **DDPM** (Diffusion) | *Unscrambling an egg* | Learn to iteratively remove noise, one small step at a time |
| **Flow Matching** | *Drawing a straight line* | Learn a velocity field that pushes noise to data along direct paths |

In [ ]:
# ============================================================
# Visualization: The Generative Modeling Goal
# ============================================================
# Let's visualize what we're trying to achieve: transform noise -> data

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel 1: Gaussian noise
noise = np.random.randn(500, 2)
axes[0].scatter(noise[:, 0], noise[:, 1], s=8, alpha=0.6, c='#6c757d')
axes[0].set_title('Noise Distribution\n$x \\sim \\mathcal{N}(0, I)$', fontweight='bold')
axes[0].set_xlim(-3.5, 3.5)
axes[0].set_ylim(-3.5, 3.5)
axes[0].set_aspect('equal')

# Panel 2: The arrow
axes[1].set_xlim(0, 10)
axes[1].set_ylim(0, 10)
axes[1].annotate('', xy=(8, 5), xytext=(2, 5),
                arrowprops=dict(arrowstyle='->', lw=3, color='#e63946'))
axes[1].text(5, 7, 'Learn this\ntransformation!', ha='center', va='center',
            fontsize=16, fontweight='bold', color='#e63946')
axes[1].text(5, 3, '$G_\\theta: \\mathcal{N}(0,I) \\to p_{data}$',
            ha='center', va='center', fontsize=14, color='#333')
axes[1].axis('off')
axes[1].set_title('Neural Network\n(what we learn)', fontweight='bold')

# Panel 3: Moon data (our target)
data = make_moons(500, noise=0.05)[0]
axes[2].scatter(data[:, 0], data[:, 1], s=8, alpha=0.6, c='#2a9d8f')
axes[2].set_title('Data Distribution\n$x \\sim p_{data}$', fontweight='bold')
axes[2].set_xlim(-1.5, 2.5)
axes[2].set_ylim(-1, 1.5)
axes[2].set_aspect('equal')

plt.suptitle('The Goal of Generative Modeling', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

<a id="part2"></a>
# Part 2: DDPM -- Denoising Diffusion Probabilistic Models

## The Key Insight: Destruction is Easy, Creation is Hard

Instead of learning the transformation in one shot, DDPM breaks it into **many tiny steps**:

1. **Forward process (destruction):** Gradually add noise to data until it becomes pure Gaussian noise. This requires no learning -- it's just math!
2. **Reverse process (creation):** Learn a neural network that reverses each tiny noise-adding step.

### The Forward Process (Adding Noise)

Given a clean data point $x_0$, we define a sequence $x_0, x_1, \ldots, x_T$ where each step adds a bit more noise:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t}\, x_{t-1},\; \beta_t I)$$

where $\beta_t$ is a small noise level that increases over time.

**The beautiful shortcut:** We can jump directly to any timestep $t$ without computing intermediate steps:

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

where $\bar{\alpha}_t = \prod_{s=1}^{t} (1 - \beta_s)$

This is the **reparameterization trick** -- it makes training efficient because we can sample any timestep directly!

In [ ]:
# ============================================================
# Visualization: The Noise Schedule
# ============================================================
# These parameters control how fast we add noise. Let's see what they look like.

n_steps = 100
betas = torch.linspace(1e-4, 0.02, n_steps)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Beta schedule
axes[0].plot(betas.numpy(), color='#e63946', linewidth=2)
axes[0].fill_between(range(n_steps), betas.numpy(), alpha=0.2, color='#e63946')
axes[0].set_xlabel('Timestep $t$')
axes[0].set_ylabel('$\\beta_t$')
axes[0].set_title('Noise Schedule $\\beta_t$\n(how much noise to add at each step)')

# Alpha bars (cumulative product)
axes[1].plot(alpha_bars.numpy(), color='#2a9d8f', linewidth=2)
axes[1].fill_between(range(n_steps), alpha_bars.numpy(), alpha=0.2, color='#2a9d8f')
axes[1].set_xlabel('Timestep $t$')
axes[1].set_ylabel('$\\bar{\\alpha}_t$')
axes[1].set_title('Signal Retention $\\bar{\\alpha}_t$\n(how much original data survives)')
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].text(n_steps*0.6, 0.55, '50% signal remaining', color='gray', fontsize=9)

# Signal vs Noise ratio
signal = torch.sqrt(alpha_bars).numpy()
noise_level = torch.sqrt(1 - alpha_bars).numpy()
axes[2].plot(signal, color='#2a9d8f', linewidth=2, label='Signal: $\\sqrt{\\bar{\\alpha}_t}$')
axes[2].plot(noise_level, color='#e63946', linewidth=2, label='Noise: $\\sqrt{1-\\bar{\\alpha}_t}$')
axes[2].fill_between(range(n_steps), signal, alpha=0.15, color='#2a9d8f')
axes[2].fill_between(range(n_steps), noise_level, alpha=0.15, color='#e63946')
axes[2].set_xlabel('Timestep $t$')
axes[2].set_title('Signal vs Noise Coefficients\n$x_t = \\sqrt{\\bar{\\alpha}_t}\\, x_0 + \\sqrt{1-\\bar{\\alpha}_t}\\, \\epsilon$')
axes[2].legend(fontsize=10)

plt.tight_layout()
plt.show()

print("Key insight: At t=0, the data is almost unchanged. At t=T, it's almost pure noise.")
print(f"  alpha_bar[0]  = {alpha_bars[0]:.4f}  (99.99% signal)")
print(f"  alpha_bar[50] = {alpha_bars[50]:.4f}  ({alpha_bars[50]*100:.1f}% signal)")
print(f"  alpha_bar[99] = {alpha_bars[99]:.4f}  ({alpha_bars[99]*100:.1f}% signal)")

In [ ]:
# ============================================================
# Visualization: Forward Diffusion Process on 2D Data
# ============================================================
# Watch how our moon-shaped data gradually dissolves into noise!

x_0 = torch.tensor(make_moons(1000, noise=0.05)[0], dtype=torch.float32)

timesteps_to_show = [0, 10, 25, 50, 75, 99]
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(20, 3.5))

for i, t in enumerate(timesteps_to_show):
    alpha_bar_t = alpha_bars[t]
    noise = torch.randn_like(x_0)
    x_t = torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1 - alpha_bar_t) * noise
    
    # Color by how much signal is preserved
    color_val = alpha_bar_t.item()
    color = plt.cm.RdYlGn(color_val)
    
    axes[i].scatter(x_t[:, 0].numpy(), x_t[:, 1].numpy(), s=3, alpha=0.5,
                   c=[color]*len(x_t))
    axes[i].set_xlim(-3.5, 3.5)
    axes[i].set_ylim(-3.5, 3.5)
    axes[i].set_aspect('equal')
    axes[i].set_title(f't = {t}\n$\\bar{{\\alpha}}_t$ = {alpha_bar_t:.3f}', fontsize=11)
    
    # Add progress bar at bottom
    bar_width = color_val
    axes[i].barh(-3.2, bar_width * 6 - 3, left=-3, height=0.15, 
                color='#2a9d8f', alpha=0.7)
    axes[i].barh(-3.2, (1-color_val) * 6, left=-3 + bar_width*6, height=0.15,
                color='#e63946', alpha=0.7)

plt.suptitle('Forward Diffusion: Data → Noise  (left to right)',
            fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Animation: Forward Diffusion Process (saved as GIF)
# ============================================================
# An animated version showing the continuous dissolution of structure

x_0_anim = torch.tensor(make_moons(500, noise=0.05)[0], dtype=torch.float32)

fig, ax = plt.subplots(figsize=(6, 6))
scatter = ax.scatter([], [], s=10, alpha=0.5)
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
title = ax.set_title('')

# Fixed noise for smooth animation
fixed_noise = torch.randn_like(x_0_anim)

def update_forward(frame):
    t = frame
    if t < len(alpha_bars):
        alpha_bar_t = alpha_bars[t]
    else:
        alpha_bar_t = alpha_bars[-1]
    x_t = torch.sqrt(alpha_bar_t) * x_0_anim + torch.sqrt(1 - alpha_bar_t) * fixed_noise
    
    # Color gradient from green (data) to red (noise)
    frac = 1 - alpha_bar_t.item()
    colors = plt.cm.RdYlGn(1 - frac)
    scatter.set_offsets(x_t.numpy())
    scatter.set_color(colors)
    title.set_text(f'Forward Diffusion: t={t}  |  Signal: {alpha_bar_t:.1%}  |  Noise: {1-alpha_bar_t:.1%}')
    return scatter, title

ani = FuncAnimation(fig, update_forward, frames=n_steps, interval=80, blit=True)
ani.save('forward_diffusion.gif', writer='pillow', fps=15)
plt.close()

display(HTML('<h4>Forward Process: Data dissolving into noise</h4>'))
display(Image(filename='forward_diffusion.gif'))

### The Reverse Process (Removing Noise)

Now comes the clever part: if we can learn to **reverse** each tiny noise-adding step, we can start from pure noise and generate data!

The reverse process is:

$$p_\theta(x_{t-1} | x_t) = \mathcal{N}\left(x_{t-1};\; \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\,\epsilon_\theta(x_t, t)\right),\; \beta_t I\right)$$

Where $\epsilon_\theta(x_t, t)$ is our neural network that **predicts the noise** that was added.

### The Training Objective

The loss is beautifully simple -- just MSE between the predicted noise and the actual noise:

$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon}\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

**Training algorithm:**
1. Sample a data point $x_0$ from the dataset
2. Sample a random timestep $t \sim \text{Uniform}(0, T)$
3. Sample noise $\epsilon \sim \mathcal{N}(0, I)$
4. Compute noisy version: $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \epsilon$
5. Predict the noise: $\hat{\epsilon} = \epsilon_\theta(x_t, t)$
6. Minimize $\|\epsilon - \hat{\epsilon}\|^2$

## Building the DDPM Model

Let's implement DDPM step by step. We'll start with 2D data (moons) to build intuition.

In [ ]:
# ============================================================
# DDPM Model Implementation (annotated for education)
# ============================================================

class DDPM(nn.Module):
    """
    Denoising Diffusion Probabilistic Model.
    
    The model has two parts:
    1. A noise schedule (betas, alphas, alpha_bars) -- fixed, no learning
    2. A neural network that predicts noise -- this is what we train
    """
    def __init__(self, dim: int = 2, h: int = 64, n_steps: int = 100):
        super().__init__()
        self.n_steps = n_steps
        
        # ---- Noise schedule (not learned) ----
        # beta_t: amount of noise added at each step (small → large)
        self.betas = torch.linspace(1e-4, 0.02, n_steps)
        # alpha_t = 1 - beta_t: fraction of signal kept at each step
        self.alphas = 1.0 - self.betas
        # alpha_bar_t = product of all alphas up to t: total signal remaining
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)
        
        # ---- Noise prediction network (learned) ----
        # Input: [timestep, noisy_data] → Output: predicted noise
        # We concatenate t (1 dim) with x_t (dim dims) = dim+1 input
        self.net = nn.Sequential(
            nn.Linear(dim + 1, h), nn.ELU(),
            nn.Linear(h, h),       nn.ELU(),
            nn.Linear(h, h),       nn.ELU(),
            nn.Linear(h, dim)      # Output same dimension as data
        )
    
    def forward(self, t: Tensor, x_t: Tensor) -> Tensor:
        """Predict the noise in x_t at timestep t."""
        t = t.view(-1, 1)  # Ensure t is [batch, 1]
        return self.net(torch.cat((t, x_t), dim=-1))
    
    @torch.no_grad()
    def sample_step(self, x_t: Tensor, t: int) -> Tensor:
        """
        One step of the reverse process: x_t → x_{t-1}
        
        Uses the formula:
        x_{t-1} = (1/sqrt(alpha_t)) * (x_t - (beta_t/sqrt(1-alpha_bar_t)) * eps_theta(x_t, t))
                   + sqrt(beta_t) * z    (where z ~ N(0,I), except at t=0)
        """
        alpha_t = self.alphas[t]
        alpha_bar_t = self.alpha_bars[t]
        
        # Coefficient for the predicted noise
        coeff = self.betas[t] / torch.sqrt(1 - alpha_bar_t)
        
        # Predict noise at this timestep
        t_normalized = torch.full((x_t.shape[0],), t / self.n_steps)
        predicted_noise = self(t_normalized, x_t)
        
        # Compute mean of x_{t-1}
        x_mean = (x_t - coeff * predicted_noise) / torch.sqrt(alpha_t)
        
        # Add stochastic noise (except at t=0)
        if t > 0:
            noise = torch.randn_like(x_t)
            return x_mean + torch.sqrt(self.betas[t]) * noise
        return x_mean

print("DDPM model architecture:")
ddpm = DDPM()
print(ddpm)
print(f"\nTotal parameters: {sum(p.numel() for p in ddpm.parameters()):,}")

### Visualizing the Network Architecture

Our noise prediction network is simple: a 4-layer MLP. It takes `[t, x_t]` as input and outputs the predicted noise $\hat{\epsilon}$.

```
Input: [t, x₁, x₂]  (3 values)
       │
  ┌────▼────┐
  │ Linear  │  3 → 64
  │  + ELU  │
  ├─────────┤
  │ Linear  │  64 → 64
  │  + ELU  │
  ├─────────┤
  │ Linear  │  64 → 64
  │  + ELU  │
  ├─────────┤
  │ Linear  │  64 → 2
  └────┬────┘
       │
Output: [ε̂₁, ε̂₂]  (predicted noise, 2 values)
```

For images, we'll later replace this MLP with a **U-Net** -- but the concept is identical!

## Training DDPM on 2D Moon Data

In [ ]:
# ============================================================
# Training DDPM on 2D Moon Data (with live loss tracking)
# ============================================================

ddpm = DDPM()
optimizer = torch.optim.Adam(ddpm.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

n_iterations = 10000
losses = []
snapshot_iters = [0, 100, 500, 2000, 5000, 9999]  # Save samples at these iterations
snapshots = {}

print("Training DDPM on 2D moon data...")
print("=" * 50)

for i in range(n_iterations):
    # Step 1: Sample data
    x_0 = Tensor(make_moons(256, noise=0.05)[0])
    
    # Step 2: Sample random timesteps
    t = torch.randint(0, ddpm.n_steps, (x_0.shape[0],))
    
    # Step 3: Sample noise
    noise = torch.randn_like(x_0)
    
    # Step 4: Create noisy version (the forward process shortcut!)
    alpha_bar_t = ddpm.alpha_bars[t].view(-1, 1)
    x_t = torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1 - alpha_bar_t) * noise
    
    # Step 5: Predict noise and compute loss
    optimizer.zero_grad()
    t_normalized = t.float() / ddpm.n_steps
    predicted_noise = ddpm(t_normalized, x_t)
    loss = loss_fn(predicted_noise, noise)
    
    # Step 6: Update weights
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    # Take generation snapshots at key iterations
    if i in snapshot_iters:
        with torch.no_grad():
            x_sample = torch.randn(300, 2)
            for step in reversed(range(ddpm.n_steps)):
                x_sample = ddpm.sample_step(x_sample, step)
            snapshots[i] = x_sample.numpy()
    
    if (i+1) % 2000 == 0:
        print(f"  Iteration {i+1:>5d}/{n_iterations}  |  Loss: {np.mean(losses[-200:]):.4f}")

print("=" * 50)
print("Training complete!")

In [ ]:
# ============================================================
# Visualization: Training Loss Curve + Generation Snapshots
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [1, 1.2]})

# --- Top: Loss curve ---
ax = axes[0]
# Smooth the loss for readability
window = 100
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
ax.plot(smoothed, color='#457b9d', linewidth=1.5, alpha=0.9)
ax.fill_between(range(len(smoothed)), smoothed, alpha=0.1, color='#457b9d')
ax.set_xlabel('Training Iteration')
ax.set_ylabel('MSE Loss')
ax.set_title('DDPM Training Loss (smoothed)', fontweight='bold')
ax.set_yscale('log')

# Mark snapshot points
for si in snapshot_iters:
    if si < len(smoothed):
        ax.axvline(x=si, color='#e63946', alpha=0.3, linestyle='--')
        ax.text(si, ax.get_ylim()[1]*0.8, f'  iter {si}', fontsize=8, color='#e63946')

# --- Bottom: Generation quality over training ---
ax2 = axes[1]
ax2.axis('off')
n_snapshots = len(snapshots)
real_data = make_moons(300, noise=0.05)[0]

for idx, (iteration, samples) in enumerate(sorted(snapshots.items())):
    # Create subplot within the bottom area
    sub_ax = fig.add_axes([0.05 + idx * 0.155, 0.02, 0.13, 0.38])
    sub_ax.scatter(samples[:, 0], samples[:, 1], s=3, alpha=0.5, c='#e63946', label='Generated')
    sub_ax.scatter(real_data[:, 0], real_data[:, 1], s=1, alpha=0.15, c='#2a9d8f', label='Real')
    sub_ax.set_xlim(-2, 3)
    sub_ax.set_ylim(-1.5, 2)
    sub_ax.set_title(f'Iter {iteration}', fontsize=9, fontweight='bold')
    sub_ax.set_xticks([])
    sub_ax.set_yticks([])
    if idx == 0:
        sub_ax.legend(fontsize=6, loc='upper right')

axes[1].set_title('Generation Quality Over Training', fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## DDPM Sampling: Watching Noise Become Data

Now let's watch the reverse process in action -- starting from pure noise and iteratively denoising to create moon-shaped data.

In [ ]:
# ============================================================
# Animation: DDPM Reverse Process (Noise → Data)
# ============================================================
%matplotlib inline

x = torch.randn(500, 2)
trajectory = [x.numpy().copy()]  # Store full trajectory for later

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
scatter = ax.scatter(x[:, 0].numpy(), x[:, 1].numpy(), s=8, alpha=0.5, c='#6c757d')
title = ax.set_title('')

# Add reference data in background
ref_data = make_moons(500, noise=0.05)[0]
ax.scatter(ref_data[:, 0], ref_data[:, 1], s=2, alpha=0.08, c='#2a9d8f')

def update_ddpm(frame):
    global x
    t = ddpm.n_steps - 1 - frame
    x = ddpm.sample_step(x, t)
    trajectory.append(x.numpy().copy())
    
    # Color shift from gray (noise) to blue (data)
    frac = frame / ddpm.n_steps
    color = plt.cm.cool(frac)
    scatter.set_offsets(x.detach().numpy())
    scatter.set_color(color)
    title.set_text(f'DDPM Reverse Process  |  Step {frame}/{ddpm.n_steps}  |  t = {t}')
    return scatter, title

ani = FuncAnimation(fig, update_ddpm, frames=ddpm.n_steps, interval=60, blit=True)
ani.save('ddpm_reverse.gif', writer='pillow', fps=20)
plt.close()

display(HTML('<h4>DDPM: Noise → Moon Data (reverse diffusion)</h4>'))
display(Image(filename='ddpm_reverse.gif'))

In [ ]:
# ============================================================
# Visualization: Particle Trajectories During DDPM Sampling
# ============================================================
# Track individual points as they move from noise to data

fig, ax = plt.subplots(figsize=(8, 8))

# Select a few particles to track
n_track = 15
track_indices = np.random.choice(500, n_track, replace=False)

# Plot trajectories
colors = plt.cm.tab20(np.linspace(0, 1, n_track))
for idx, pidx in enumerate(track_indices):
    traj_x = [trajectory[step][pidx, 0] for step in range(0, len(trajectory), 2)]
    traj_y = [trajectory[step][pidx, 1] for step in range(0, len(trajectory), 2)]
    ax.plot(traj_x, traj_y, color=colors[idx], alpha=0.5, linewidth=0.8)
    # Mark start (noise) and end (data)
    ax.scatter(traj_x[0], traj_y[0], color=colors[idx], marker='x', s=40, zorder=5)
    ax.scatter(traj_x[-1], traj_y[-1], color=colors[idx], marker='o', s=30, zorder=5)

# Background: final generated points
final = trajectory[-1]
ax.scatter(final[:, 0], final[:, 1], s=3, alpha=0.15, c='#2a9d8f')

ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
ax.set_title('DDPM Particle Trajectories\n(x = start in noise, o = end in data)', fontweight='bold')
ax.legend(['Particle path'], loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

print("Notice how the paths are curved and stochastic -- DDPM takes a winding route from noise to data.")

<a id="part3"></a>
# Part 3: Flow Matching -- A Simpler Path

## The Idea: Why Not Just Draw a Straight Line?

DDPM works great, but notice how the particle trajectories were **curved and stochastic**. Flow Matching takes a more direct approach:

> **Instead of learning to remove noise iteratively, learn a velocity field that pushes noise toward data along straight paths.**

### The Mathematical Setup

Given a noise sample $x_0 \sim \mathcal{N}(0, I)$ and a data sample $x_1 \sim p_\text{data}$, we define a **straight interpolation path**:

$$x_t = (1 - t)\, x_0 + t\, x_1, \quad t \in [0, 1]$$

The **velocity** along this path is simply:

$$\frac{dx_t}{dt} = x_1 - x_0$$

We train a neural network $v_\theta(x_t, t)$ to predict this velocity:

$$\mathcal{L} = \mathbb{E}_{t, x_0, x_1}\left[\|v_\theta(x_t, t) - (x_1 - x_0)\|^2\right]$$

At inference, we start from noise and follow the learned velocity field using an ODE solver.

### DDPM vs Flow Matching at a Glance

| | **DDPM** | **Flow Matching** |
|---|---|---|
| **Path** | Curved (stochastic) | Straight |
| **What the network predicts** | Added noise $\epsilon$ | Velocity $dx/dt$ |
| **Time direction** | $t: T \to 0$ (backward) | $t: 0 \to 1$ (forward) |
| **Sampling** | Stochastic (SDE) | Deterministic (ODE) |
| **Typical # steps** | 100-1000 | 10-100 (faster!) |

In [ ]:
# ============================================================
# Visualization: Flow Matching Interpolation Paths
# ============================================================
# Show the straight paths that flow matching uses vs DDPM's curved paths

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left: Flow Matching straight paths ---
ax = axes[0]
n_paths = 12
x_0_paths = np.random.randn(n_paths, 2)  # noise
x_1_paths = make_moons(n_paths, noise=0.05)[0]  # data

colors_paths = plt.cm.viridis(np.linspace(0.2, 0.8, n_paths))
for i in range(n_paths):
    ts = np.linspace(0, 1, 50)
    path_x = (1 - ts) * x_0_paths[i, 0] + ts * x_1_paths[i, 0]
    path_y = (1 - ts) * x_0_paths[i, 1] + ts * x_1_paths[i, 1]
    ax.plot(path_x, path_y, color=colors_paths[i], alpha=0.7, linewidth=1.5)
    ax.scatter(x_0_paths[i, 0], x_0_paths[i, 1], color=colors_paths[i], 
              marker='x', s=60, zorder=5, linewidths=2)
    ax.scatter(x_1_paths[i, 0], x_1_paths[i, 1], color=colors_paths[i],
              marker='o', s=60, zorder=5, edgecolors='black', linewidths=0.5)
    # Arrow showing velocity direction
    mid_idx = len(ts) // 2
    ax.annotate('', xy=(path_x[mid_idx+2], path_y[mid_idx+2]),
               xytext=(path_x[mid_idx], path_y[mid_idx]),
               arrowprops=dict(arrowstyle='->', color=colors_paths[i], lw=1.5))

ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
ax.set_title('Flow Matching: Straight Paths\n$x_t = (1-t)x_0 + tx_1$', fontweight='bold', fontsize=13)
ax.set_xlabel('x marks noise (t=0), circles mark data (t=1)')

# --- Right: Show x_t at different times ---
ax = axes[1]
times_show = [0.0, 0.25, 0.5, 0.75, 1.0]
n_pts = 400
x_0_batch = np.random.randn(n_pts, 2)
x_1_batch = make_moons(n_pts, noise=0.05)[0]

for t_val in times_show:
    x_t = (1 - t_val) * x_0_batch + t_val * x_1_batch
    alpha = 0.3 + 0.4 * t_val  # more visible near data
    ax.scatter(x_t[:, 0], x_t[:, 1], s=5, alpha=alpha,
              label=f't = {t_val:.2f}', c=plt.cm.plasma(t_val))

ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
ax.set_title('Distribution at Different Times\n(noise → data as t goes 0→1)', fontweight='bold', fontsize=13)
ax.legend(fontsize=10, markerscale=3)

plt.tight_layout()
plt.show()

## Building the Flow Matching Model

The model architecture is very similar to DDPM -- the difference is in **what we predict** and **how we sample**.

In [ ]:
# ============================================================
# Flow Matching Model Implementation (annotated)
# ============================================================

class Flow(nn.Module):
    """
    Flow Matching model.
    
    Key differences from DDPM:
    - No noise schedule needed (no betas, alphas, etc.)
    - Predicts velocity (dx/dt) instead of noise (epsilon)
    - Uses ODE integration for sampling instead of stochastic steps
    """
    def __init__(self, dim: int = 2, h: int = 64):
        super().__init__()
        # Same architecture as DDPM! The magic is in the training target.
        self.net = nn.Sequential(
            nn.Linear(dim + 1, h), nn.ELU(),
            nn.Linear(h, h),       nn.ELU(),
            nn.Linear(h, h),       nn.ELU(),
            nn.Linear(h, dim)
        )
    
    def forward(self, t: Tensor, x_t: Tensor) -> Tensor:
        """Predict the velocity field v(x_t, t)."""
        return self.net(torch.cat((t, x_t), dim=-1))
    
    @torch.no_grad()
    def step(self, x_t: Tensor, t_start: Tensor, t_end: Tensor) -> Tensor:
        """
        One integration step using the Midpoint method (2nd order Runge-Kutta).
        
        More accurate than Euler's method:
        - Euler:    x_{t+dt} = x_t + dt * v(x_t, t)
        - Midpoint: x_{t+dt} = x_t + dt * v(x_mid, t_mid)
          where x_mid = x_t + dt/2 * v(x_t, t), t_mid = t + dt/2
        """
        t_start = t_start.view(1, 1).expand(x_t.shape[0], 1)
        dt = t_end - t_start
        
        # Midpoint method: evaluate velocity at the middle of the interval
        v_start = self(t=t_start, x_t=x_t)
        x_mid = x_t + v_start * dt / 2
        t_mid = t_start + dt / 2
        v_mid = self(t=t_mid, x_t=x_mid)
        
        return x_t + dt * v_mid

print("Flow model architecture:")
flow = Flow()
print(flow)
print(f"\nTotal parameters: {sum(p.numel() for p in flow.parameters()):,}")
print("\nNotice: Same architecture as DDPM! The difference is in training & sampling.")

## Training Flow Matching

The training loop is even simpler than DDPM -- no noise schedule to manage!

In [ ]:
# ============================================================
# Training Flow Matching on 2D Moon Data
# ============================================================

flow = Flow()
optimizer = torch.optim.Adam(flow.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

n_iterations = 10000
flow_losses = []
flow_snapshots = {}
flow_snapshot_iters = [0, 100, 500, 2000, 5000, 9999]

print("Training Flow Matching on 2D moon data...")
print("=" * 50)

for i in range(n_iterations):
    # Step 1: Sample data (x_1) and noise (x_0)
    x_1 = Tensor(make_moons(256, noise=0.05)[0])
    x_0 = torch.randn_like(x_1)
    
    # Step 2: Sample random time t ∈ [0, 1]
    t = torch.rand(len(x_1), 1)
    
    # Step 3: Interpolate between noise and data
    x_t = (1 - t) * x_0 + t * x_1
    
    # Step 4: Target velocity is simply x_1 - x_0 (the direction from noise to data)
    target_velocity = x_1 - x_0
    
    # Step 5: Predict velocity and compute loss
    optimizer.zero_grad()
    predicted_velocity = flow(t=t, x_t=x_t)
    loss = loss_fn(predicted_velocity, target_velocity)
    loss.backward()
    optimizer.step()
    
    flow_losses.append(loss.item())
    
    # Take generation snapshots
    if i in flow_snapshot_iters:
        with torch.no_grad():
            x_sample = torch.randn(300, 2)
            n_sample_steps = 100
            ts = torch.linspace(0, 1.0, n_sample_steps)
            for s in range(n_sample_steps - 1):
                x_sample = flow.step(x_sample, ts[s], ts[s+1])
            flow_snapshots[i] = x_sample.numpy()
    
    if (i+1) % 2000 == 0:
        print(f"  Iteration {i+1:>5d}/{n_iterations}  |  Loss: {np.mean(flow_losses[-200:]):.4f}")

print("=" * 50)
print("Training complete!")

In [ ]:
# ============================================================
# Visualization: Learned Velocity Field (Quiver Plot)
# ============================================================
# This is a powerful visualization unique to flow matching --
# we can visualize the velocity field the model learned at different times.

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
times_to_show = [0.0, 0.25, 0.5, 0.75]

for idx, t_val in enumerate(times_to_show):
    ax = axes[idx]
    
    # Create a grid of points
    grid_size = 20
    x_range = np.linspace(-3, 3, grid_size)
    y_range = np.linspace(-3, 3, grid_size)
    xx, yy = np.meshgrid(x_range, y_range)
    grid_points = torch.tensor(np.stack([xx.ravel(), yy.ravel()], axis=1), dtype=torch.float32)
    
    # Evaluate velocity field at grid points
    t_grid = torch.full((grid_points.shape[0], 1), t_val)
    with torch.no_grad():
        velocities = flow(t=t_grid, x_t=grid_points).numpy()
    
    # Compute speed for coloring
    speed = np.sqrt(velocities[:, 0]**2 + velocities[:, 1]**2)
    
    # Quiver plot
    q = ax.quiver(xx.ravel(), yy.ravel(), velocities[:, 0], velocities[:, 1],
                  speed, cmap='coolwarm', alpha=0.8, scale=80)
    
    # Show data distribution at this time
    n_bg = 300
    x_0_bg = np.random.randn(n_bg, 2)
    x_1_bg = make_moons(n_bg, noise=0.05)[0]
    x_t_bg = (1 - t_val) * x_0_bg + t_val * x_1_bg
    ax.scatter(x_t_bg[:, 0], x_t_bg[:, 1], s=3, alpha=0.2, c='black')
    
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.set_title(f't = {t_val:.2f}', fontweight='bold', fontsize=13)

plt.suptitle('Learned Velocity Field at Different Times\n(arrows show where the model pushes points)',
            fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("The arrows show the velocity v(x, t) at each point.")
print("Notice how the field organizes to push scattered noise toward the moon shape.")

In [ ]:
# ============================================================
# Animation: Flow Matching Sampling (Noise → Data via ODE)
# ============================================================
%matplotlib inline

x_flow = torch.randn(500, 2)
flow_trajectory = [x_flow.numpy().copy()]
n_flow_steps = 100
time_steps_flow = torch.linspace(0, 1.0, n_flow_steps)

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
scatter = ax.scatter(x_flow[:, 0].numpy(), x_flow[:, 1].numpy(), s=8, alpha=0.5, c='#6c757d')
title = ax.set_title('')

# Reference data in background
ax.scatter(ref_data[:, 0], ref_data[:, 1], s=2, alpha=0.08, c='#2a9d8f')

def update_flow(frame):
    global x_flow
    if frame < n_flow_steps - 1:
        x_flow = flow.step(x_flow, time_steps_flow[frame], time_steps_flow[frame + 1])
    flow_trajectory.append(x_flow.numpy().copy())
    
    frac = frame / n_flow_steps
    color = plt.cm.plasma(frac)
    scatter.set_offsets(x_flow.detach().numpy())
    scatter.set_color(color)
    title.set_text(f'Flow Matching ODE Integration  |  t = {time_steps_flow[frame]:.2f}')
    return scatter, title

ani = FuncAnimation(fig, update_flow, frames=n_flow_steps, interval=60, blit=True)
ani.save('flow_sampling.gif', writer='pillow', fps=20)
plt.close()

display(HTML('<h4>Flow Matching: Noise → Moon Data (ODE integration)</h4>'))
display(Image(filename='flow_sampling.gif'))

In [ ]:
# ============================================================
# Visualization: DDPM vs Flow Matching -- Side-by-Side Comparison
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# --- Row 1: Loss curves ---
ax = axes[0, 0]
window = 100
ddpm_smooth = np.convolve(losses, np.ones(window)/window, mode='valid')
flow_smooth = np.convolve(flow_losses, np.ones(window)/window, mode='valid')
ax.plot(ddpm_smooth, label='DDPM', color='#457b9d', linewidth=1.5)
ax.plot(flow_smooth, label='Flow Matching', color='#e76f51', linewidth=1.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.set_title('Training Loss Comparison', fontweight='bold')
ax.legend()
ax.set_yscale('log')

# --- Row 1: Trajectory comparison ---
ax = axes[0, 1]
n_track = 10
track_idx = np.random.choice(500, n_track, replace=False)
colors_t = plt.cm.Set1(np.linspace(0, 1, n_track))

for i, pidx in enumerate(track_idx):
    traj_x = [trajectory[s][pidx, 0] for s in range(0, len(trajectory), 3)]
    traj_y = [trajectory[s][pidx, 1] for s in range(0, len(trajectory), 3)]
    ax.plot(traj_x, traj_y, color=colors_t[i], alpha=0.4, linewidth=0.8)

ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
ax.set_title('DDPM Trajectories\n(curved, stochastic)', fontweight='bold')

ax = axes[0, 2]
for i, pidx in enumerate(track_idx):
    traj_x = [flow_trajectory[s][pidx, 0] for s in range(0, len(flow_trajectory), 3)]
    traj_y = [flow_trajectory[s][pidx, 1] for s in range(0, len(flow_trajectory), 3)]
    ax.plot(traj_x, traj_y, color=colors_t[i], alpha=0.4, linewidth=0.8)

ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
ax.set_title('Flow Trajectories\n(straighter, deterministic)', fontweight='bold')

# --- Row 2: Final samples comparison ---
real_data = make_moons(500, noise=0.05)[0]

ax = axes[1, 0]
ax.scatter(real_data[:, 0], real_data[:, 1], s=5, alpha=0.5, c='#2a9d8f')
ax.set_xlim(-2, 3)
ax.set_ylim(-1.5, 2)
ax.set_title('Real Data', fontweight='bold')
ax.set_aspect('equal')

ax = axes[1, 1]
ddpm_final = snapshots[9999]
ax.scatter(ddpm_final[:, 0], ddpm_final[:, 1], s=5, alpha=0.5, c='#457b9d')
ax.scatter(real_data[:, 0], real_data[:, 1], s=2, alpha=0.1, c='#2a9d8f')
ax.set_xlim(-2, 3)
ax.set_ylim(-1.5, 2)
ax.set_title('DDPM Generated', fontweight='bold')
ax.set_aspect('equal')

ax = axes[1, 2]
flow_final = flow_snapshots[9999]
ax.scatter(flow_final[:, 0], flow_final[:, 1], s=5, alpha=0.5, c='#e76f51')
ax.scatter(real_data[:, 0], real_data[:, 1], s=2, alpha=0.1, c='#2a9d8f')
ax.set_xlim(-2, 3)
ax.set_ylim(-1.5, 2)
ax.set_title('Flow Matching Generated', fontweight='bold')
ax.set_aspect('equal')

plt.suptitle('DDPM vs Flow Matching: Head-to-Head', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Visualization: Effect of Number of Sampling Steps
# ============================================================
# One key advantage of flow matching: it works well with fewer steps

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
step_counts = [5, 10, 25, 50, 100]

for idx, n_s in enumerate(step_counts):
    torch.manual_seed(42)
    x_test = torch.randn(400, 2)
    ts = torch.linspace(0, 1.0, n_s)
    
    with torch.no_grad():
        for s in range(n_s - 1):
            x_test = flow.step(x_test, ts[s], ts[s+1])
    
    axes[idx].scatter(x_test[:, 0].numpy(), x_test[:, 1].numpy(), s=5, alpha=0.5, c='#e76f51')
    axes[idx].scatter(real_data[:, 0], real_data[:, 1], s=1, alpha=0.1, c='#2a9d8f')
    axes[idx].set_xlim(-2, 3)
    axes[idx].set_ylim(-1.5, 2)
    axes[idx].set_aspect('equal')
    axes[idx].set_title(f'{n_s} steps', fontweight='bold', fontsize=13)

plt.suptitle('Flow Matching: Quality vs Number of Sampling Steps\n(fewer steps = faster generation)',
            fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("Flow matching can produce reasonable results with very few steps!")
print("This is a major practical advantage over DDPM.")

<a id="part4"></a>
# Part 4: Fine-Tuning Stable Diffusion on Pokemon

## From 2D Toys to Real Image Generation with Stable Diffusion

Now we move from toy examples to a **production-quality** generative model. Instead of training a small model from scratch (which produces blurry, low-quality results), we will **fine-tune Stable Diffusion 1.5** -- the same architecture behind many modern image generation tools.

### Stable Diffusion Architecture

Stable Diffusion has four main components:

```
                        ┌─────────────────────────────────────────────────┐
                        │           Stable Diffusion Pipeline             │
                        │                                                 │
  "a blue pokemon"      │  ┌──────────┐                                  │
  ──────────────────────┼─►│  CLIP     │  text features                  │
                        │  │  Text     │──────────┐                      │
                        │  │  Encoder  │          │ cross-attention       │
                        │  └──────────┘          ▼                       │
                        │                  ┌───────────┐                 │
  Random noise          │                  │   U-Net   │  denoised       │   ┌───────┐
  (64×64×4 latent) ─────┼─────────────────►│  + LoRA   │──latent────────┼──►│  VAE   │──► 512×512
                        │                  │  adapters │                 │   │Decoder │    image
                        │    timestep ────►│           │                 │   └───────┘
                        │                  └───────────┘                 │
                        └─────────────────────────────────────────────────┘
```

| Component | Role |
|-----------|------|
| **VAE** (Variational Autoencoder) | Compresses 512x512 images into 64x64x4 latent space (8x compression). Diffusion happens in this compressed space -- much cheaper than pixel space! |
| **CLIP Text Encoder** | Converts text prompts into feature vectors that guide the U-Net via cross-attention. Frozen (not trained). |
| **U-Net** | The denoiser -- predicts and removes noise from latent representations, conditioned on text. This is what we fine-tune with LoRA. |
| **Noise Scheduler** | Controls the noise schedule (same concept as our Part 2 DDPM scheduler). |

### Why Fine-Tune Instead of Train from Scratch?

Training a diffusion model from scratch on ~800 images produces poor results. Stable Diffusion was pretrained on **2 billion** image-text pairs -- it already knows how to generate realistic images. We just need to teach it the **Pokemon style** using **LoRA** (Low-Rank Adaptation), which:
- Trains only **~1-4M parameters** instead of all 860M
- Takes **~15-30 minutes** on a single GPU
- Produces **high-quality 512x512 Pokemon images**

| Component | Choice | Why |
|-----------|--------|-----|
| **Base model** | Stable Diffusion 1.5 | Proven, smallest SD model (~4GB fp16), rich fine-tuning ecosystem |
| **Fine-tuning** | LoRA | Efficient (~0.2% of params), fast training, great results |
| **Resolution** | 512x512 | Native SD 1.5 resolution (latent space: 64x64x4) |
| **Dataset** | Pokemon BLIP Captions | 833 stylized images with simple captions -- perfect for demonstrating fine-tuning |

## Step 1: Load the Pokemon Dataset

We use the **Pokemon BLIP Captions** dataset -- it's ideal for teaching because:
- Captions are simple and visually grounded ("a blue pokemon with large eyes")
- Stylized images mean even a small model can produce recognizable outputs
- Students can quickly see whether conditioning is working

In [ ]:
# ============================================================
# Step 1: Load and Explore the Pokemon Dataset
# ============================================================
from datasets import load_dataset
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image as PILImage

# Load Pokemon BLIP Captions dataset
print("Loading Pokemon BLIP Captions dataset...")
pokemon_dataset = load_dataset("reach-vb/pokemon-blip-captions", split="train")
print(f"Dataset size: {len(pokemon_dataset)} images")
print(f"Features: {pokemon_dataset.features}")
print(f"\nExample captions:")
for i in range(5):
    print(f"  [{i}] {pokemon_dataset[i]['text']}")

In [ ]:
# ============================================================
# Visualization: Dataset Gallery
# ============================================================
# Let's look at what we're working with!

fig, axes = plt.subplots(3, 6, figsize=(18, 10))

for i, ax in enumerate(axes.flat):
    idx = np.random.randint(len(pokemon_dataset))
    item = pokemon_dataset[idx]
    img = item['image']
    caption = item['text']
    
    ax.imshow(img)
    ax.set_title(caption[:40] + ('...' if len(caption) > 40 else ''),
                fontsize=8, wrap=True)
    ax.axis('off')

plt.suptitle('Pokemon BLIP Captions Dataset -- Sample Gallery',
            fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Step 1b: Create PyTorch Dataset with Preprocessing
# ============================================================

IMAGE_SIZE = 512  # Native resolution for Stable Diffusion 1.5

# Image preprocessing: resize to 512x512, normalize to [-1, 1]
image_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

class PokemonCaptionDataset(Dataset):
    """Wraps the HuggingFace dataset into a PyTorch Dataset."""
    def __init__(self, hf_dataset, transform):
        self.dataset = hf_dataset
        self.transform = transform
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = self.transform(item['image'].convert('RGB'))
        caption = item['text']
        return image, caption

train_dataset = PokemonCaptionDataset(pokemon_dataset, image_transform)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True,
                         num_workers=0, drop_last=True)

# Verify a batch
images, captions = next(iter(train_loader))
print(f"Image batch shape: {images.shape}  (batch, channels, height, width)")
print(f"Image value range: [{images.min():.1f}, {images.max():.1f}]  (normalized to [-1, 1])")
print(f"Caption example: '{captions[0]}'")
print(f"\nDataset: {len(train_dataset)} images at {IMAGE_SIZE}x{IMAGE_SIZE}")

In [ ]:
# ============================================================
# Visualization: VAE Encode / Decode Demo
# ============================================================
# Stable Diffusion works in LATENT space, not pixel space.
# The VAE compresses 512x512 RGB images into 64x64x4 latent representations.
# Let's see this in action!

from diffusers import AutoencoderKL

# Load just the VAE for now (we'll load the full pipeline later)
vae_demo = AutoencoderKL.from_pretrained(
    "runwayml/stable-diffusion-v1-5", subfolder="vae"
).to(device)
vae_demo.eval()
vae_demo.requires_grad_(False)

# Take a few images from the dataset
n_demo = 4
demo_images = []
demo_captions = []
for idx in [0, 50, 100, 200]:
    item = pokemon_dataset[idx]
    img = image_transform(item['image'].convert('RGB'))
    demo_images.append(img)
    demo_captions.append(item['text'])
demo_batch = torch.stack(demo_images).to(device)

# Encode to latent space, then decode back
with torch.no_grad():
    latents = vae_demo.encode(demo_batch).latent_dist.sample()
    latents_scaled = latents * vae_demo.config.scaling_factor
    decoded = vae_demo.decode(latents_scaled / vae_demo.config.scaling_factor).sample

print(f"Original images:  {demo_batch.shape}  (batch, 3, 512, 512)")
print(f"Latent space:     {latents.shape}  (batch, 4, 64, 64)  -- 8x spatial compression!")
print(f"Decoded images:   {decoded.shape}  (batch, 3, 512, 512)")
print(f"\nCompression ratio: {512*512*3 / (64*64*4):.1f}x fewer values in latent space")

fig, axes = plt.subplots(3, n_demo, figsize=(5 * n_demo, 14))

for i in range(n_demo):
    # Original
    orig = (demo_batch[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
    axes[0, i].imshow(orig)
    axes[0, i].set_title(demo_captions[i][:35], fontsize=9)
    axes[0, i].axis('off')

    # Latent (show first 3 of 4 channels as RGB-ish visualization)
    lat_viz = latents[i, :3].cpu().permute(1, 2, 0).numpy()
    lat_viz = (lat_viz - lat_viz.min()) / (lat_viz.max() - lat_viz.min() + 1e-8)
    axes[1, i].imshow(lat_viz)
    axes[1, i].set_title(f'Latent: {latents.shape[2]}x{latents.shape[3]}x{latents.shape[1]}', fontsize=10)
    axes[1, i].axis('off')

    # Reconstructed
    recon = (decoded[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
    axes[2, i].imshow(recon)
    axes[2, i].set_title('VAE Reconstruction', fontsize=10)
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Original\n512x512', fontsize=12, rotation=0, labelpad=60, va='center')
axes[1, 0].set_ylabel('Latent\n64x64x4', fontsize=12, rotation=0, labelpad=60, va='center')
axes[2, 0].set_ylabel('Decoded\n512x512', fontsize=12, rotation=0, labelpad=60, va='center')

plt.suptitle('VAE: Image → Latent → Reconstructed Image\nDiffusion happens in the compressed 64x64x4 latent space',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

del vae_demo
if device.type == 'cuda':
    torch.cuda.empty_cache()

## Step 2: Load Pretrained Stable Diffusion 1.5

We load the full Stable Diffusion 1.5 pipeline and extract its four components:

1. **`vae`** -- compresses images to/from latent space (frozen, not trained)
2. **`text_encoder`** + **`tokenizer`** -- converts text to CLIP embeddings (frozen, not trained)
3. **`unet`** -- the denoiser that we will fine-tune with LoRA
4. **`noise_scheduler`** -- controls the diffusion noise schedule

All components except the U-Net's LoRA adapters remain frozen during fine-tuning.

In [ ]:
# ============================================================
# Step 2: Load Pretrained Stable Diffusion 1.5
# ============================================================
from diffusers import StableDiffusionPipeline, DDPMScheduler
from transformers import CLIPTextModel, CLIPTokenizer

MODEL_ID = "runwayml/stable-diffusion-v1-5"

print(f"Loading Stable Diffusion 1.5 from: {MODEL_ID}")
print("This may take a minute on first download (~4GB)...\n")

# Load the full pipeline, then extract components
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32
)

# Extract the four core components
vae = pipe.vae.to(device)
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder.to(device)
unet = pipe.unet.to(device)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

# Freeze VAE and text encoder -- we only fine-tune the UNet (via LoRA)
vae.requires_grad_(False)
vae.eval()
text_encoder.requires_grad_(False)
text_encoder.eval()

# Print component summary
print("Stable Diffusion 1.5 Components:")
print("=" * 55)
vae_params = sum(p.numel() for p in vae.parameters())
te_params = sum(p.numel() for p in text_encoder.parameters())
unet_params = sum(p.numel() for p in unet.parameters())
print(f"  VAE:            {vae_params:>12,} params (frozen)")
print(f"  Text Encoder:   {te_params:>12,} params (frozen)")
print(f"  U-Net:          {unet_params:>12,} params (will add LoRA)")
print(f"  {'─'*43}")
print(f"  Total:          {vae_params+te_params+unet_params:>12,} params")
print(f"\n  Text hidden dim: {text_encoder.config.hidden_size}")
print(f"  Latent channels: {vae.config.latent_channels}")
print(f"  Latent scale:    {vae.config.scaling_factor}")

del pipe
if device.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Baseline: Generate with vanilla SD 1.5 (BEFORE fine-tuning)
# ============================================================

baseline_prompts = [
    "a drawing of a green pokemon with red eyes",
    "a blue water creature with fins",
    "a red fire-breathing dragon pokemon",
    "a cute yellow electric mouse pokemon",
]

@torch.no_grad()
def generate_sd(unet, vae, text_encoder, tokenizer, noise_scheduler,
                prompts, device, guidance_scale=7.5, num_steps=30, seed=42):
    """Generate images with Stable Diffusion components."""
    generator = torch.Generator(device=device).manual_seed(seed)
    batch_size = len(prompts)
    weight_dtype = next(unet.parameters()).dtype

    tokens = tokenizer(prompts, padding="max_length",
                       max_length=tokenizer.model_max_length,
                       truncation=True, return_tensors="pt").to(device)
    text_emb = text_encoder(tokens.input_ids)[0].to(dtype=weight_dtype)

    uncond_tokens = tokenizer([""] * batch_size, padding="max_length",
                              max_length=tokenizer.model_max_length,
                              return_tensors="pt").to(device)
    uncond_emb = text_encoder(uncond_tokens.input_ids)[0].to(dtype=weight_dtype)

    latents = torch.randn(batch_size, 4, 64, 64,
                          device=device, dtype=weight_dtype, generator=generator)
    noise_scheduler.set_timesteps(num_steps, device=device)
    latents = latents * noise_scheduler.init_noise_sigma

    for t in noise_scheduler.timesteps:
        latent_input = torch.cat([latents] * 2)
        latent_input = noise_scheduler.scale_model_input(latent_input, t)
        noise_pred = unet(latent_input, t,
                          encoder_hidden_states=torch.cat([uncond_emb, text_emb])).sample
        noise_uncond, noise_cond = noise_pred.chunk(2)
        noise_pred = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
        latents = noise_scheduler.step(noise_pred, t, latents).prev_sample

    latents_dec = latents / vae.config.scaling_factor
    images = vae.decode(latents_dec.to(vae.dtype)).sample
    images = (images * 0.5 + 0.5).clamp(0, 1)
    return images.float().cpu()

print("Generating baseline images with vanilla SD 1.5 (before fine-tuning)...")
baseline_images = generate_sd(unet, vae, text_encoder, tokenizer,
                              noise_scheduler, baseline_prompts, device)

fig, axes = plt.subplots(1, len(baseline_prompts), figsize=(5 * len(baseline_prompts), 6))
for i, (img, prompt) in enumerate(zip(baseline_images, baseline_prompts)):
    axes[i].imshow(img.permute(1, 2, 0).numpy())
    axes[i].set_title(prompt[:45], fontsize=9, wrap=True)
    axes[i].axis('off')

plt.suptitle('Baseline: Vanilla SD 1.5 (before Pokemon fine-tuning)\nThe model generates images but not in the Pokemon art style',
            fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Step 3: LoRA -- Efficient Fine-Tuning

### The Problem with Full Fine-Tuning

The U-Net has **~860M parameters**. Fine-tuning all of them would require enormous GPU memory and risk overfitting on our small 833-image dataset.

### The LoRA Solution

**LoRA (Low-Rank Adaptation)** freezes the original weights and injects small trainable matrices into the attention layers:

$$W' = W + BA$$

where $W$ is the frozen original weight matrix $(d \times d)$, and $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times d}$ are the small trainable matrices with rank $r \ll d$.

```
Original attention layer:          With LoRA:
                                    
  x ──► [  W  ] ──► y              x ──► [  W (frozen) ] ──► y
         (d×d)                      │                         ▲
                                    └──► [ B ]──►[ A ] ──────┘
                                          (d×r)   (r×d)
                                          trainable!
```

**Key insight:** Most of the "knowledge" needed for Pokemon style can be captured in a low-rank update. With rank r=32, we inject LoRA into both the attention layers *and* the feed-forward layers of the transformer blocks, training ~6M parameters instead of 860M -- less than 1% of the model.

Cross-attention is how the U-Net "reads" the text:

```
U-Net spatial features          CLIP text features
(what it's generating)          (what to generate)
        │                              │
        ▼                              ▼
    ┌───────┐                     ┌───────┐
    │  Q    │    attention        │ K, V  │
    │(query)│◄───────────────────►│(key,  │
    │       │   "Which text       │ value)│
    └───┬───┘    tokens matter    └───────┘
        │         for this pixel?"
        ▼
  Updated spatial features
  (now informed by text)
```

At each spatial position, the U-Net asks: *"Which parts of the text description are relevant here?"*
- A pixel in the sky might attend to "blue"
- A pixel near the face might attend to "large eyes"

In [ ]:
# ============================================================
# Step 3: Inject LoRA Adapters into the U-Net
# ============================================================
from peft import LoraConfig, get_peft_model

# LoRA configuration: which layers to adapt and with what rank
lora_config = LoraConfig(
    r=32,                             # Rank of the low-rank matrices (higher = more capacity)
    lora_alpha=32,                    # Scaling factor (alpha/r = effective learning rate scale)
    init_lora_weights="gaussian",
    target_modules=[                  # Inject LoRA into attention AND feed-forward layers
        "to_k", "to_q", "to_v", "to_out.0",
        "ff.net.0.proj", "ff.net.2",  # Feed-forward layers in transformer blocks
    ],
)

# Inject LoRA adapters into the U-Net
unet = get_peft_model(unet, lora_config)

# Count parameters
total_params = sum(p.numel() for p in unet.parameters())
trainable_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print("LoRA injection complete!")
print("=" * 55)
print(f"  Total U-Net params:     {total_params:>12,}")
print(f"  Frozen params:          {frozen_params:>12,}  ({frozen_params/total_params:.1%})")
print(f"  Trainable LoRA params:  {trainable_params:>12,}  ({trainable_params/total_params:.1%})")
print(f"\n  → We only train {trainable_params/total_params:.2%} of the model!")
print(f"  → LoRA rank = {lora_config.r}, targeting {len(lora_config.target_modules)} layer types")

In [ ]:
# ============================================================
# Visualization: LoRA Concept Diagram
# ============================================================
# Show how LoRA injects small trainable matrices into frozen attention layers

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel 1: Standard vs LoRA weight update ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Standard Fine-Tuning vs LoRA', fontweight='bold', fontsize=13)

# Standard fine-tuning
rect_std = mpatches.FancyBboxPatch((0.3, 6), 4, 3, boxstyle="round,pad=0.15",
                                    facecolor='#e63946', alpha=0.2, edgecolor='#e63946', lw=2)
ax.add_patch(rect_std)
ax.text(2.3, 8.3, 'Standard Fine-Tuning', ha='center', fontsize=11, fontweight='bold')
ax.text(2.3, 7.3, 'Update ALL 860M params', ha='center', fontsize=10)
ax.text(2.3, 6.5, 'Slow, high memory', ha='center', fontsize=9, color='gray')

# LoRA
rect_lora = mpatches.FancyBboxPatch((5.5, 6), 4.2, 3, boxstyle="round,pad=0.15",
                                     facecolor='#2a9d8f', alpha=0.2, edgecolor='#2a9d8f', lw=2)
ax.add_patch(rect_lora)
ax.text(7.6, 8.3, 'LoRA Fine-Tuning', ha='center', fontsize=11, fontweight='bold')
ax.text(7.6, 7.3, f'Update only ~{trainable_params/1e6:.1f}M params', ha='center', fontsize=10)
ax.text(7.6, 6.5, 'Fast, low memory', ha='center', fontsize=9, color='gray')

# Formula
ax.text(5, 4.5, "W' = W + B A", ha='center', fontsize=18, fontweight='bold',
        family='monospace')
ax.text(5, 3.3, 'W: frozen original weights (d × d)', ha='center', fontsize=10)
ax.text(5, 2.5, 'B: trainable (d × r)', ha='center', fontsize=10, color='#2a9d8f')
ax.text(5, 1.8, 'A: trainable (r × d)', ha='center', fontsize=10, color='#2a9d8f')
ax.text(5, 0.8, f'r = {lora_config.r} (rank) << d = 320...1280', ha='center',
        fontsize=10, style='italic', color='gray')

# --- Panel 2: Where LoRA is injected ---
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('LoRA Injection Points in U-Net', fontweight='bold', fontsize=13)

layers = ['to_q (Query)', 'to_k (Key)', 'to_v (Value)', 'to_out (Output)']
colors_l = ['#457b9d', '#e76f51', '#2a9d8f', '#f4a261']
for i, (name, c) in enumerate(zip(layers, colors_l)):
    y = 8 - i * 2
    # Frozen weight box
    rect = mpatches.FancyBboxPatch((0.5, y - 0.5), 3, 1, boxstyle="round,pad=0.1",
                                    facecolor='#ddd', alpha=0.5, edgecolor='gray', lw=1.5)
    ax2.add_patch(rect)
    ax2.text(2, y, 'W (frozen)', ha='center', va='center', fontsize=9, color='gray')

    # Plus sign
    ax2.text(4.2, y, '+', ha='center', va='center', fontsize=16, fontweight='bold')

    # LoRA adapter box
    rect_a = mpatches.FancyBboxPatch((5, y - 0.5), 3.5, 1, boxstyle="round,pad=0.1",
                                      facecolor=c, alpha=0.25, edgecolor=c, lw=2)
    ax2.add_patch(rect_a)
    ax2.text(6.75, y, f'LoRA: {name}', ha='center', va='center', fontsize=9,
            fontweight='bold', color=c)

ax2.text(5, 0.3, f'Applied to every cross-attention & self-attention block in the U-Net',
        ha='center', fontsize=9, style='italic', color='gray')

plt.tight_layout()
plt.show()


## Step 4: Training with Classifier-Free Guidance Dropout

### What is Classifier-Free Guidance (CFG)?

During **training**, we randomly drop the text conditioning ~10% of the time (replace with empty text). This teaches the model to generate *both* with and without text guidance.

During **inference**, we use the trick:

$$\hat{\epsilon} = \epsilon_\theta(x_t, \varnothing) + s \cdot \left(\epsilon_\theta(x_t, \text{text}) - \epsilon_\theta(x_t, \varnothing)\right)$$

where $s$ is the **guidance scale**:
- $s = 1$: No guidance (use text prediction as-is)
- $s = 3\text{-}6$: Moderate guidance (better text following)
- $s > 7$: Strong guidance (very prompt-faithful, may lose diversity)

Think of it as: *"How strongly should the model listen to the text?"*

In [ ]:
# ============================================================
# Visualization: Classifier-Free Guidance Explained
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Training with CFG dropout
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Training diagram
ax.text(5, 9, 'Training with CFG Dropout', ha='center', fontsize=13, fontweight='bold')

# 90% path
rect1 = mpatches.FancyBboxPatch((0.5, 5.5), 4, 2.5, boxstyle="round,pad=0.1",
                                  facecolor='#2a9d8f', alpha=0.2, edgecolor='#2a9d8f')
ax.add_patch(rect1)
ax.text(2.5, 7.5, '90% of time:', ha='center', fontsize=10, fontweight='bold')
ax.text(2.5, 6.5, 'Use real caption\n"a blue pokemon..."', ha='center', fontsize=9)

# 10% path
rect2 = mpatches.FancyBboxPatch((5.5, 5.5), 4, 2.5, boxstyle="round,pad=0.1",
                                  facecolor='#e63946', alpha=0.2, edgecolor='#e63946')
ax.add_patch(rect2)
ax.text(7.5, 7.5, '10% of time:', ha='center', fontsize=10, fontweight='bold')
ax.text(7.5, 6.5, 'Use empty text ""\n(unconditional)', ha='center', fontsize=9)

ax.text(5, 4.5, 'Both feed into same U-Net', ha='center', fontsize=10, style='italic')
ax.annotate('', xy=(5, 3.5), xytext=(5, 4.2),
           arrowprops=dict(arrowstyle='->', lw=2))
rect3 = mpatches.FancyBboxPatch((2, 2), 6, 1.5, boxstyle="round,pad=0.1",
                                  facecolor='#457b9d', alpha=0.2, edgecolor='#457b9d')
ax.add_patch(rect3)
ax.text(5, 2.75, 'U-Net learns both modes', ha='center', fontsize=10, fontweight='bold')

# Panel 2: Inference formula
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

ax.text(5, 9, 'Inference with CFG', ha='center', fontsize=13, fontweight='bold')
ax.text(5, 7.5, 'Two forward passes:', ha='center', fontsize=11)
ax.text(5, 6.5, r'$\epsilon_{uncond} = \epsilon_\theta(x_t, \varnothing)$', ha='center', fontsize=12)
ax.text(5, 5.5, r'$\epsilon_{cond} = \epsilon_\theta(x_t, \mathrm{text})$', ha='center', fontsize=12)
ax.text(5, 4, 'Combine:', ha='center', fontsize=11, fontweight='bold')
ax.text(5, 2.8, r'$\hat{\epsilon} = \epsilon_{uncond} + s \cdot (\epsilon_{cond} - \epsilon_{uncond})$',
       ha='center', fontsize=13, color='#e63946',
       bbox=dict(boxstyle='round', facecolor='#fee', edgecolor='#e63946'))
ax.text(5, 1.5, 'guidance scale s controls\nhow much to follow the text',
       ha='center', fontsize=10, style='italic')

# Panel 3: Effect of guidance scale
ax = axes[2]
scales = [1, 3, 5, 7, 10]
quality = [0.3, 0.7, 0.9, 0.85, 0.6]  # Conceptual
diversity = [0.9, 0.7, 0.5, 0.3, 0.15]
text_follow = [0.2, 0.5, 0.8, 0.95, 0.99]

ax.plot(scales, quality, 'o-', color='#2a9d8f', linewidth=2, markersize=8, label='Image quality')
ax.plot(scales, diversity, 's-', color='#457b9d', linewidth=2, markersize=8, label='Diversity')
ax.plot(scales, text_follow, '^-', color='#e63946', linewidth=2, markersize=8, label='Text following')
ax.axvline(x=5, color='gray', linestyle='--', alpha=0.5)
ax.text(5.2, 0.95, 'sweet spot', fontsize=9, color='gray')
ax.set_xlabel('Guidance Scale (s)', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Effect of Guidance Scale', fontweight='bold', fontsize=13)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.1)

plt.suptitle('Classifier-Free Guidance: The Key Trick for Text-Conditioned Generation',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Helper: Encode text and images for Stable Diffusion training
# ============================================================

def encode_text_sd(captions, tokenizer, text_encoder, device):
    """Convert text captions into CLIP hidden states for cross-attention."""
    tokens = tokenizer(
        captions,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        text_features = text_encoder(tokens.input_ids)[0]
    return text_features


def encode_images_to_latents(images, vae):
    """Encode pixel-space images into VAE latent space."""
    with torch.no_grad():
        latent_dist = vae.encode(images.to(vae.dtype)).latent_dist
        latents = latent_dist.sample() * vae.config.scaling_factor
    return latents


# Quick test
test_captions = ["a blue pokemon with large eyes"]
test_features = encode_text_sd(test_captions, tokenizer, text_encoder, device)
print(f"Text features shape: {test_features.shape}  (batch, seq_len, hidden_dim={test_features.shape[-1]})")

test_latents = encode_images_to_latents(images[:1].to(device), vae)
print(f"Image latents shape: {test_latents.shape}  (batch, 4, 64, 64)")
print(f"\nThe U-Net will denoise these 64x64x4 latents, conditioned on the {test_features.shape[-1]}-dim text features.")


In [ ]:
# ============================================================
# Helper: Generate samples for progress monitoring
# ============================================================

@torch.no_grad()
def generate_samples_lora(unet, vae, text_encoder, tokenizer, noise_scheduler,
                          prompts, device, guidance_scale=7.5, num_steps=30, seed=42):
    """Generate images with the LoRA-fine-tuned Stable Diffusion."""
    generator = torch.Generator(device=device).manual_seed(seed)
    batch_size = len(prompts)
    weight_dtype = torch.float16 if device.type == 'cuda' else torch.float32

    # Encode text
    text_emb = encode_text_sd(prompts, tokenizer, text_encoder, device).to(dtype=weight_dtype)
    uncond_emb = encode_text_sd([""] * batch_size, tokenizer, text_encoder, device).to(dtype=weight_dtype)

    # Start from random latent noise
    latents = torch.randn(batch_size, 4, 64, 64,
                          device=device, dtype=weight_dtype, generator=generator)
    noise_scheduler.set_timesteps(num_steps, device=device)
    latents = latents * noise_scheduler.init_noise_sigma

    for t in noise_scheduler.timesteps:
        latent_input = torch.cat([latents] * 2)
        latent_input = noise_scheduler.scale_model_input(latent_input, t)
        noise_pred = unet(latent_input, t,
                          encoder_hidden_states=torch.cat([uncond_emb, text_emb])).sample
        noise_uncond, noise_cond = noise_pred.chunk(2)
        noise_pred = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
        latents = noise_scheduler.step(noise_pred, t, latents).prev_sample

    # Decode latents to pixel images
    latents_dec = latents / vae.config.scaling_factor
    images = vae.decode(latents_dec.to(vae.dtype)).sample
    images = (images * 0.5 + 0.5).clamp(0, 1)
    return images.float().cpu()


def show_images(images, prompts=None, ncols=4, figsize=None):
    """Display a grid of generated images with optional captions."""
    n = len(images)
    nrows = (n + ncols - 1) // ncols
    if figsize is None:
        figsize = (4 * ncols, 4.5 * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    if nrows == 1:
        axes = [axes] if ncols == 1 else list(axes)
    else:
        axes = [ax for row in axes for ax in row]
    for i in range(n):
        img = images[i].permute(1, 2, 0).numpy() if images[i].dim() == 3 else images[i]
        axes[i].imshow(img)
        if prompts:
            axes[i].set_title(prompts[i][:45], fontsize=8, wrap=True)
        axes[i].axis('off')
    for j in range(n, len(axes)):
        axes[j].axis('off')
    return fig

print("Helper functions defined: generate_samples_lora(), show_images()")


## Step 5: LoRA Fine-Tuning Training Loop

The training loop is conceptually identical to Part 2's DDPM, but now in **latent space**:

1. **Encode** images to VAE latents (64x64x4) using frozen VAE
2. **Encode** text captions with frozen CLIP (with 10% CFG dropout for unconditional generation)
3. **Cast** inputs to fp32 for the UNet forward pass (gradient precision matters for small LoRA weights)
4. **Sample** a random timestep and add noise to the latents
5. **Predict** the noise with the U-Net (conditioned on text via cross-attention)
6. **Compute** MSE loss between predicted and actual noise
7. **Update** only the LoRA parameters via AdamW with **cosine LR schedule + warmup**

**Training recipe highlights:**
- **1500 steps** with gradient accumulation of 4 (effective batch size = 4)
- **Cosine LR decay** with 100-step linear warmup prevents early instability and enables continued learning
- **fp32 UNet forward pass** ensures full gradient precision for the small LoRA weight updates
- **Low weight decay (1e-4)** avoids pushing the small LoRA weights back toward zero

In [ ]:
# ============================================================
# Step 5: LoRA Fine-Tuning Training Loop
# ============================================================
import math

weight_dtype = torch.float16 if device.type == 'cuda' else torch.float32

# Move frozen models to lower precision for memory efficiency
vae.to(dtype=weight_dtype)
text_encoder.to(dtype=weight_dtype)

# Keep UNet in fp32 for gradient precision on LoRA weights
unet.float()

# Training config
num_steps = 1500
gradient_accumulation_steps = 4
base_lr = 5e-5
warmup_steps = 100
cfg_dropout_prob = 0.10
log_every = 50
sample_every = 300

optimizer = torch.optim.AdamW(
    [p for p in unet.parameters() if p.requires_grad],
    lr=base_lr, weight_decay=1e-4, betas=(0.9, 0.999), eps=1e-8
)

# Cosine LR schedule with linear warmup
def get_lr(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / max(1, num_steps - warmup_steps)
    return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))

lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)

test_prompts = [
    "a drawing of a green pokemon with red eyes",
    "a red fire-breathing dragon pokemon",
    "a cute pink round creature with big eyes",
    "a blue water pokemon with fins",
]

train_losses = []
lr_history = []
sample_history = {}
global_step = 0

# Pre-compute empty-text embedding for CFG dropout
empty_emb = encode_text_sd([""], tokenizer, text_encoder, device).to(dtype=weight_dtype)

trainable_count = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print("Starting LoRA fine-tuning...")
print(f"  Total steps:          {num_steps}")
print(f"  Grad accumulation:    {gradient_accumulation_steps}")
print(f"  Effective batch size: {train_loader.batch_size * gradient_accumulation_steps}")
print(f"  Base LR:              {base_lr}")
print(f"  LR warmup steps:      {warmup_steps}")
print(f"  LR schedule:          cosine decay (min 5% of base)")
print(f"  Weight decay:         1e-4")
print(f"  CFG dropout:          {cfg_dropout_prob:.0%}")
print(f"  Trainable params:     {trainable_count:,}")
print("=" * 60)

unet.train()
data_iter = iter(train_loader)
optimizer.zero_grad()
running_loss = 0.0

for step in range(num_steps * gradient_accumulation_steps):
    try:
        batch_images, batch_captions = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        batch_images, batch_captions = next(data_iter)

    batch_images = batch_images.to(device, dtype=weight_dtype)

    # 1. Encode images to latent space
    with torch.no_grad():
        latents = encode_images_to_latents(batch_images, vae)

    # 2. Encode text (with CFG dropout)
    with torch.no_grad():
        if np.random.random() < cfg_dropout_prob:
            text_emb = empty_emb.expand(latents.shape[0], -1, -1)
        else:
            text_emb = encode_text_sd(
                list(batch_captions), tokenizer, text_encoder, device
            ).to(dtype=weight_dtype)

    # Cast to fp32 for UNet forward pass (LoRA weights are fp32)
    latents_f32 = latents.float()
    text_emb_f32 = text_emb.float()

    # 3. Sample noise and timesteps
    noise = torch.randn_like(latents_f32)
    timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                              (latents_f32.shape[0],), device=device).long()

    # 4. Add noise to latents (forward process)
    noisy_latents = noise_scheduler.add_noise(latents_f32, noise, timesteps)

    # 5. Predict the noise (in fp32 for gradient precision)
    noise_pred = unet(noisy_latents, timesteps,
                      encoder_hidden_states=text_emb_f32).sample

    # 6. Compute loss and accumulate gradients
    loss = F.mse_loss(noise_pred.float(), noise.float()) / gradient_accumulation_steps
    loss.backward()
    running_loss += loss.item()

    # Update weights after accumulation
    if (step + 1) % gradient_accumulation_steps == 0:
        torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        global_step += 1

        train_losses.append(running_loss)
        lr_history.append(optimizer.param_groups[0]['lr'])
        running_loss = 0.0

        if global_step % log_every == 0:
            avg_loss = np.mean(train_losses[-log_every:])
            current_lr = optimizer.param_groups[0]['lr']
            print(f"  Step {global_step:>5d}/{num_steps}  |  Loss: {avg_loss:.4f}  |  LR: {current_lr:.2e}")

        if global_step % sample_every == 0 or global_step == num_steps:
            unet.eval()
            imgs = generate_samples_lora(
                unet, vae, text_encoder, tokenizer, noise_scheduler,
                test_prompts, device, guidance_scale=7.5, num_steps=30
            )
            sample_history[global_step] = imgs
            unet.train()
            print(f"         → Saved sample images at step {global_step}")

    if global_step >= num_steps:
        break

print("=" * 60)
print(f"Training complete! Final avg loss: {np.mean(train_losses[-50:]):.4f}")


In [ ]:
# ============================================================
# Visualization: Training Loss Curve + Learning Rate Schedule
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# --- Loss curve ---
ax1.plot(range(1, len(train_losses)+1), train_losses, color='#457b9d', linewidth=1, alpha=0.3, label='Raw')

if len(train_losses) > 20:
    window = min(30, len(train_losses) // 5)
    smoothed = np.convolve(train_losses, np.ones(window)/window, mode='valid')
    ax1.plot(range(window, len(train_losses)+1), smoothed, color='#457b9d', linewidth=2.5, label='Smoothed')

ax1.set_xlabel('Training Step')
ax1.set_ylabel('MSE Loss')
ax1.set_title('LoRA Fine-Tuning Loss Curve', fontweight='bold', fontsize=14)
ax1.legend()

for ep in sample_history:
    ax1.axvline(x=ep, color='#e63946', alpha=0.3, linestyle='--')
    ax1.text(ep, ax1.get_ylim()[1]*0.95, f' step {ep}', fontsize=7, color='#e63946')

# --- Learning rate schedule ---
if lr_history:
    ax2.plot(range(1, len(lr_history)+1), lr_history, color='#e76f51', linewidth=2)
    ax2.axvline(x=warmup_steps, color='gray', alpha=0.5, linestyle='--', label=f'Warmup ends ({warmup_steps})')
    ax2.set_xlabel('Training Step')
    ax2.set_ylabel('Learning Rate')
    ax2.set_title('Learning Rate Schedule (Warmup + Cosine)', fontweight='bold', fontsize=14)
    ax2.legend()
    ax2.ticklabel_format(axis='y', style='scientific', scilimits=(-4,-4))

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Visualization: Generation Quality Over Training
# ============================================================

if sample_history:
    epochs_sorted = sorted(sample_history.keys())
    n_epochs_shown = len(epochs_sorted)
    n_prompts = len(test_prompts)

    fig, axes = plt.subplots(n_prompts, n_epochs_shown,
                             figsize=(4 * n_epochs_shown, 4.5 * n_prompts))
    if n_prompts == 1:
        axes = axes[np.newaxis, :]
    if n_epochs_shown == 1:
        axes = axes[:, np.newaxis]

    for col, step_num in enumerate(epochs_sorted):
        imgs = sample_history[step_num]
        for row in range(n_prompts):
            ax = axes[row, col]
            img = imgs[row].permute(1, 2, 0).numpy()
            ax.imshow(img)
            ax.axis('off')
            if row == 0:
                ax.set_title(f'Step {step_num}', fontweight='bold', fontsize=12)
            if col == 0:
                ax.set_ylabel(test_prompts[row][:30] + '...', fontsize=8, rotation=0,
                             labelpad=100, va='center')

    plt.suptitle('Generation Quality During LoRA Fine-Tuning\n(rows = prompts, columns = training steps)',
                fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No sample history available (training may not have run yet).")


## Step 6: Inference with Fine-Tuned Model

Now let's use our LoRA-fine-tuned Stable Diffusion to generate Pokemon images from text prompts. We'll explore:
1. **Gallery generation** -- various Pokemon prompts
2. **Before vs After** -- comparing vanilla SD 1.5 with our fine-tuned model
3. **Guidance scale** -- controlling how closely the model follows the prompt
4. **Denoising process** -- watching a Pokemon emerge from noise step by step
5. **Seed variation** -- same prompt, different random seeds

In [ ]:
# ============================================================
# Inference: Generate Pokemon Gallery
# ============================================================

unet.eval()

gallery_prompts = [
    "a drawing of a green pokemon with red eyes",
    "a red dragon breathing fire",
    "a blue water pokemon with fins and a tail",
    "a cute yellow electric mouse pokemon",
    "a purple ghost-like pokemon floating",
    "a white and blue ice creature",
    "a brown furry pokemon with big ears",
    "a pink small cute round creature",
]

print("Generating Pokemon images with fine-tuned LoRA model...")
gallery_images = generate_samples_lora(
    unet, vae, text_encoder, tokenizer, noise_scheduler,
    gallery_prompts, device, guidance_scale=7.5, num_steps=30
)

fig = show_images(gallery_images, gallery_prompts, ncols=4)
fig.suptitle('Generated Pokemon (LoRA Fine-Tuned SD 1.5, guidance_scale=7.5)',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Comparison: Before vs After LoRA Fine-Tuning
# ============================================================
# The key demonstration: same prompts, dramatically different results

compare_prompts = [
    "a drawing of a green pokemon with red eyes",
    "a blue water creature with fins",
    "a red fire-breathing dragon pokemon",
    "a cute yellow electric mouse pokemon",
]

print("Generating before/after comparison...")

# Generate with fine-tuned model (same seed as baseline for fair comparison)
finetuned_images = generate_samples_lora(
    unet, vae, text_encoder, tokenizer, noise_scheduler,
    compare_prompts, device, guidance_scale=7.5, num_steps=30, seed=42
)

# Regenerate baseline (disable LoRA temporarily)
unet.disable_adapter_layers()
baseline_compare = generate_sd(
    unet, vae, text_encoder, tokenizer, noise_scheduler,
    compare_prompts, device, guidance_scale=7.5, num_steps=30, seed=42
)
unet.enable_adapter_layers()

# Display side by side
n = len(compare_prompts)
fig, axes = plt.subplots(2, n, figsize=(5 * n, 11))

for i in range(n):
    # Before (baseline)
    axes[0, i].imshow(baseline_compare[i].permute(1, 2, 0).numpy())
    axes[0, i].set_title(compare_prompts[i][:40], fontsize=9, wrap=True)
    axes[0, i].axis('off')

    # After (fine-tuned)
    axes[1, i].imshow(finetuned_images[i].permute(1, 2, 0).numpy())
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Before\n(Vanilla SD 1.5)', fontsize=12, rotation=0, labelpad=80, va='center')
axes[1, 0].set_ylabel('After\n(LoRA Fine-Tuned)', fontsize=12, rotation=0, labelpad=80, va='center')

plt.suptitle('Before vs After LoRA Fine-Tuning on Pokemon\n(same prompts, same seed)',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Experiment: Effect of Guidance Scale
# ============================================================

prompt = "a blue water pokemon with large eyes and fins"
guidance_scales = [1.0, 3.0, 5.0, 7.5, 12.0]

fig, axes = plt.subplots(1, len(guidance_scales), figsize=(5 * len(guidance_scales), 6))

print(f"Generating '{prompt}' at different guidance scales...")

for idx, gs in enumerate(guidance_scales):
    imgs = generate_samples_lora(
        unet, vae, text_encoder, tokenizer, noise_scheduler,
        [prompt], device, guidance_scale=gs, num_steps=30, seed=42
    )
    axes[idx].imshow(imgs[0].permute(1, 2, 0).numpy())
    axes[idx].set_title(f'scale = {gs}', fontweight='bold', fontsize=13)
    axes[idx].axis('off')

plt.suptitle(f'Classifier-Free Guidance Scale Comparison\nPrompt: "{prompt}"',
            fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("\nObservation:")
print("  scale=1:   Ignores the text (unconditional generation)")
print("  scale=3-5: Moderate text following")
print("  scale=7.5: Good balance of quality and prompt adherence")
print("  scale=12+: Very prompt-faithful but may over-saturate")


In [ ]:
# ============================================================
# Visualization: Step-by-Step Denoising Process
# ============================================================
# Watch as a Pokemon emerges from noise in latent space

prompt_for_anim = "a red dragon pokemon with wings"
print(f"Generating denoising timeline for: '{prompt_for_anim}'")

weight_dtype = torch.float16 if device.type == 'cuda' else torch.float32
text_emb = encode_text_sd([prompt_for_anim], tokenizer, text_encoder, device).to(dtype=weight_dtype)
uncond_emb = encode_text_sd([""], tokenizer, text_encoder, device).to(dtype=weight_dtype)

torch.manual_seed(123)
latents = torch.randn(1, 4, 64, 64, device=device, dtype=weight_dtype)
noise_scheduler.set_timesteps(30, device=device)
latents = latents * noise_scheduler.init_noise_sigma

denoising_frames = []

with torch.no_grad():
    for step_idx, t in enumerate(noise_scheduler.timesteps):
        latent_input = torch.cat([latents] * 2)
        latent_input = noise_scheduler.scale_model_input(latent_input, t)
        noise_pred = unet(latent_input, t,
                          encoder_hidden_states=torch.cat([uncond_emb, text_emb])).sample
        noise_uncond, noise_cond = noise_pred.chunk(2)
        noise_pred = noise_uncond + 7.5 * (noise_cond - noise_uncond)
        latents = noise_scheduler.step(noise_pred, t, latents).prev_sample

        # Decode to pixel space for visualization
        lat_dec = latents / vae.config.scaling_factor
        img = vae.decode(lat_dec.to(vae.dtype)).sample
        img = (img[0].float() * 0.5 + 0.5).clamp(0, 1).cpu().permute(1, 2, 0).numpy()
        denoising_frames.append((img, t.item()))

# Show as filmstrip
frames_to_show = list(range(0, len(denoising_frames), max(1, len(denoising_frames)//8)))
if (len(denoising_frames)-1) not in frames_to_show:
    frames_to_show.append(len(denoising_frames)-1)
n_frames = len(frames_to_show)

fig, axes = plt.subplots(1, n_frames, figsize=(3.5 * n_frames, 4.5))

for i, fidx in enumerate(frames_to_show):
    img, t_val = denoising_frames[fidx]
    axes[i].imshow(img)
    axes[i].set_title(f'Step {fidx+1}/{len(denoising_frames)}\nt={t_val:.0f}', fontsize=10)
    axes[i].axis('off')

plt.suptitle(f'Denoising Timeline: "{prompt_for_anim}"\nPure noise (left) → Generated Pokemon (right)',
            fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Experiment: Same Prompt, Different Random Seeds
# ============================================================

prompt_diversity = "a cute round blue pokemon"
n_samples = 6

fig, axes = plt.subplots(1, n_samples, figsize=(5 * n_samples, 6))

print(f"Generating {n_samples} variations of: '{prompt_diversity}'")

for i in range(n_samples):
    imgs = generate_samples_lora(
        unet, vae, text_encoder, tokenizer, noise_scheduler,
        [prompt_diversity], device, guidance_scale=7.5, num_steps=30, seed=i*42
    )
    axes[i].imshow(imgs[0].permute(1, 2, 0).numpy())
    axes[i].set_title(f'seed = {i * 42}', fontsize=11)
    axes[i].axis('off')

plt.suptitle(f'Same Prompt, Different Seeds → Diverse Outputs\n"{prompt_diversity}"',
            fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("Each image starts from different random noise but follows the same text guidance.")


---

# Summary

## What We Built Today

| Part | What We Did | Key Takeaway |
|------|------------|-------------|
| **Part 1** | Understood the generative modeling goal | Transform simple noise into complex data |
| **Part 2** | Built DDPM from scratch on 2D data | Iterative denoising with learned noise prediction |
| **Part 3** | Built Flow Matching from scratch | Straight paths + velocity fields = simpler & faster |
| **Part 4** | Fine-tuned Stable Diffusion with LoRA on Pokemon | Real-world text-to-image with efficient adaptation |

## Stable Diffusion Architecture We Used

| Component | Detail |
|-----------|--------|
| **Base model** | Stable Diffusion 1.5 (`runwayml/stable-diffusion-v1-5`) |
| **Resolution** | 512x512 pixels (64x64x4 latent space) |
| **VAE** | Compresses images 8x spatially (frozen) |
| **Text encoder** | CLIP ViT-L/14, 768-dim features (frozen) |
| **U-Net** | ~860M params, only LoRA adapters trained (~1-4M) |
| **LoRA rank** | r=4, targeting attention projection layers |
| **Training** | ~800 steps, batch 1 with grad accumulation 4, lr=1e-4 |
| **Dataset** | Pokemon BLIP Captions (833 images) |

## From Toy to Production

| | Part 2 (2D DDPM) | Part 4 (Stable Diffusion + LoRA) |
|---|---|---|
| **Data** | 2D moon points | 512x512 Pokemon images |
| **Space** | Direct (2D) | VAE latent space (64x64x4) |
| **Denoiser** | 4-layer MLP (~13K params) | U-Net (~860M params, ~2M trained) |
| **Conditioning** | None | CLIP text encoder + cross-attention |
| **Inference trick** | None | Classifier-free guidance |
| **Core algorithm** | DDPM | Same DDPM -- just at scale! |

## Key References

- Ho et al. (2020): *Denoising Diffusion Probabilistic Models* -- the DDPM paper
- Lipman et al. (2023): *Flow Matching for Generative Modeling* -- flow matching theory
- Rombach et al. (2022): *High-Resolution Image Synthesis with Latent Diffusion Models* -- Stable Diffusion
- Ho & Salimans (2022): *Classifier-Free Diffusion Guidance* -- the CFG trick
- Hu et al. (2022): *LoRA: Low-Rank Adaptation of Large Language Models* -- the LoRA method

## Where to Go Next

1. **SDXL / SD 3**: Larger models with better quality (requires more GPU memory)
2. **DreamBooth**: Personalize generation to specific subjects from a few images
3. **ControlNet**: Add spatial conditioning (pose, edges, depth maps) to guided generation
4. **Textual Inversion**: Learn new "concepts" as text embeddings
5. **Flow Matching for Images**: Apply flow matching (Part 3) to full image generation